# 🧠 03 — Model Training: Two-Phase Transfer Learning
**Project:** Diabetic Retinopathy Detection

**Goal:** Train EfficientNetB3 with:
- Phase 1: Frozen backbone → warm up classifier head (10 epochs)
- Phase 2: Full fine-tuning → optimize entire network (40 epochs)

---

## 📦 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB3
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
import albumentations as A

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 5
SEED        = 42
DATA_DIR    = Path('../data/processed')
MODEL_DIR   = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', [g.name for g in tf.config.list_physical_devices('GPU')])

## 📂 2. Data Pipeline with tf.data

In [ ]:
def ben_graham_preprocess(img, sigma=10):
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    return img

def load_and_preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    # Standard ImageNet normalization
    mean = tf.constant([0.485, 0.456, 0.406])
    std  = tf.constant([0.229, 0.224, 0.225])
    img  = (img - mean) / std
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def make_dataset(df, augment=False, shuffle=False):
    paths   = tf.constant(df['image_path'].values)
    labels  = tf.constant(df['diagnosis'].values)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(lambda p, l: load_and_preprocess(p, l, augment),
                          num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=SEED)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')

train_ds = make_dataset(train_df, augment=True,  shuffle=True)
val_ds   = make_dataset(val_df,   augment=False, shuffle=False)

class_weights_arr = compute_class_weight('balanced', classes=np.arange(5), y=train_df['diagnosis'].values)
class_weight_dict = dict(enumerate(class_weights_arr))

print(f'Train batches : {len(train_ds)}')
print(f'Val batches   : {len(val_ds)}')
print(f'Class weights : {class_weight_dict}')

## 🏗️ 3. Model Architecture

In [ ]:
def build_model(trainable_backbone=False):
    # Load pretrained EfficientNetB3
    backbone = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = trainable_backbone

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_model(trainable_backbone=False)
model.summary(line_length=80)

## 🚀 4. Phase 1 — Train Classifier Head (Frozen Backbone)

In [ ]:
# Custom QWK metric callback
from sklearn.metrics import cohen_kappa_score

class QWKCallback(keras.callbacks.Callback):
    def __init__(self, val_dataset, val_df):
        self.val_dataset = val_dataset
        self.val_df = val_df
        self.qwk_scores = []

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.val_dataset, verbose=0)
        pred_classes = np.argmax(preds, axis=1)
        true_classes = self.val_df['diagnosis'].values[:len(pred_classes)]
        qwk = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')
        self.qwk_scores.append(qwk)
        print(f'  → Epoch {epoch+1:02d}: QWK = {qwk:.4f}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

qwk_cb = QWKCallback(val_ds, val_df)

phase1_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(str(MODEL_DIR / 'best_phase1.keras'),
                               monitor='val_accuracy', save_best_only=True),
    qwk_cb
]

print('=== PHASE 1: Frozen backbone — training classifier head ===')
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=phase1_callbacks,
    verbose=1
)

## 🔓 5. Phase 2 — Full Fine-Tuning (Unfreeze Backbone)

In [ ]:
# Unfreeze all backbone layers
model.layers[1].trainable = True  # backbone is layers[1]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100x lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

qwk_cb2 = QWKCallback(val_ds, val_df)

phase2_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3,
                                min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint(str(MODEL_DIR / 'best_model.keras'),
                               monitor='val_accuracy', save_best_only=True, verbose=1),
    qwk_cb2
]

print('=== PHASE 2: Full fine-tuning — all layers trainable ===')
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight_dict,
    callbacks=phase2_callbacks,
    verbose=1
)

## 📈 6. Training Curves

In [ ]:
def plot_training(h1, h2, metric='accuracy'):
    p1_vals  = h1.history[metric]
    p1_vvals = h1.history[f'val_{metric}']
    p2_vals  = h2.history[metric]
    p2_vvals = h2.history[f'val_{metric}']

    all_train = p1_vals + p2_vals
    all_val   = p1_vvals + p2_vvals
    epochs    = range(1, len(all_train) + 1)
    split_ep  = len(p1_vals)

    plt.figure(figsize=(12, 5))
    plt.plot(epochs, all_train, 'b-o', markersize=3, label='Train')
    plt.plot(epochs, all_val,   'r-o', markersize=3, label='Validation')
    plt.axvline(x=split_ep + 0.5, color='gray', linestyle='--', label='Phase 1 → Phase 2')
    plt.title(f'Training & Validation {metric.capitalize()} — Two-Phase Fine-Tuning',
              fontsize=13, fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel(metric.capitalize())
    plt.legend(); plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(f'../results/figures/training_curves_{metric}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training(history_p1, history_p2, 'accuracy')
plot_training(history_p1, history_p2, 'loss')

# QWK over epochs
all_qwk = qwk_cb.qwk_scores + qwk_cb2.qwk_scores
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(all_qwk) + 1), all_qwk, 'g-o', markersize=4)
plt.axvline(x=len(qwk_cb.qwk_scores) + 0.5, color='gray', linestyle='--', label='Phase boundary')
plt.title('Quadratic Weighted Kappa (QWK) Over Training', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('QWK Score'); plt.grid(True, alpha=0.4); plt.legend()
plt.tight_layout()
plt.savefig('../results/figures/qwk_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best QWK: {max(all_qwk):.4f} at epoch {np.argmax(all_qwk)+1}')

## ✅ 7. Summary

| Phase | Epochs | Best Val Acc | Best QWK |
|-------|--------|--------------|----------|
| Phase 1 (frozen) | 10 | — | — |
| Phase 2 (full)   | 40 | — | — |

*(Fill in actual values after training)*

Model saved to `models/best_model.keras`

**Next step:** `04_Evaluation.ipynb` — confusion matrix, per-class metrics, GradCAM visualizations.